# 🧪 LocateAnything-3B · Đếm SẢN PHẨM theo VẠCH (supervision)

Đếm **cắt vạch** trên 3 video thật, gán nhãn bằng **supervision**.

| Video | Đếm | Prompt (danh từ đơn giản) | Vạch |
|---|---|---|---|
| `packages_rollers.mp4` | thùng carton | **`carton box`** | ngang y≈0.65 |
| `packages_belt.mp4` | kiện hàng | **`package`** | ngang y≈0.60 |
| `tomatoes_sorting.mp4` | cà chua | **`tomato`** | ngang y≈0.72 |

⚠️ **LocateAnything detect tốt nhất với DANH TỪ ĐƠN GIẢN tiếng Anh** (carton box /
package / box / tomato / fruit). **KHÔNG** dùng mô tả dài ("a sealed shipping
package") hay tiếng Việt ("cà chua") — LA sẽ bắt SÓT (đó là lý do lần trước ra ít/0).
Cell cuối cho **so sánh vài prompt** để chọn cái bắt nhiều nhất.

In [ ]:
# ⚙️ Cài đặt: clone repo (kèm video) + phụ thuộc + kiểm tra GPU
import os, sys, subprocess
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # giảm phân mảnh VRAM (đặt TRƯỚC khi torch nạp)

def sh(*a):
    print("$", " ".join(a)); subprocess.run(list(a), check=True)

if os.path.isdir("/kaggle/working"): WORK = "/kaggle/working"
elif os.path.isdir("/content"):      WORK = "/content"
else:                                 WORK = os.getcwd()
os.chdir(WORK); print("WORK =", WORK)

BRANCH = "claude/locate-anything-test-suite-xwju2f"
URL    = "https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git"
REPO   = os.path.join(WORK, "VisionOS")
if not os.path.isdir(os.path.join(REPO, ".git")):
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO)
else:
    sh("git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH)
    sh("git", "-C", REPO, "reset", "--hard", "origin/" + BRANCH)

CODE = os.path.join(REPO, "VisionOS")          # code + sample_videos/ ở đây
os.chdir(CODE); sys.path.insert(0, CODE)
print("CODE =", CODE)

# LocateAnything-3B CẦN transformers==4.57.1 (Colab hay có 5.x → cài ĐÚNG bản này, ĐỪNG Stop)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.1", "accelerate", "supervision",
                "eva-decord", "lmdb"], check=False)

for m in [m for m in list(sys.modules)
          if m.split(".")[0] in ("la_counting", "recognition", "run_la_conveyor")]:
    del sys.modules[m]

try:
    import torch
    print("CUDA:", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print("torch:", e)

In [ ]:
# 🎯 Cấu hình 3 video (VẠCH + prompt ĐƠN GIẢN) + xem trước vạch (không cần GPU)
import cv2, numpy as np, matplotlib.pyplot as plt

VID = os.path.join(CODE, "sample_videos")
VIDEOS = [
  {"name": "Package · con lăn", "path": os.path.join(VID, "packages_rollers.mp4"),
   "orient": "horizontal", "line_pos": 0.65,
   "query": "carton box", "alts": ["package", "box"]},        # prompt đơn giản
  {"name": "Package · có nhãn", "path": os.path.join(VID, "packages_belt.mp4"),
   "orient": "horizontal", "line_pos": 0.60,
   "query": "package", "alts": ["carton box", "box"]},
  {"name": "Cà chua · phân loại", "path": os.path.join(VID, "tomatoes_sorting.mp4"),
   "orient": "horizontal", "line_pos": 0.72,
   "query": "tomato", "alts": ["fruit", "red fruit"]},
]

# NÚM tốc độ / chất lượng
PROC_WIDTH     = 768
MAX_FRAMES     = 45
STRIDE         = 1      # =1 để ByteTrack bám track (đừng tăng)
MAX_NEW_TOKENS = 512

def _prev(v, frac=0.5):
    cap = cv2.VideoCapture(v["path"]); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * frac)); ok, fr = cap.read(); cap.release()
    if not ok: return None
    h, w = fr.shape[:2]; y = int(v["line_pos"] * h)
    cv2.line(fr, (0, y), (w, y), (0, 0, 255), 3)
    return cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, len(VIDEOS), figsize=(16, 4))
for ax, v in zip(np.atleast_1d(axes), VIDEOS):
    im = _prev(v)
    if im is not None: ax.imshow(im)
    ax.set_title(v["name"] + f"\nprompt='{v['query']}'"); ax.axis("off")
plt.tight_layout(); plt.show()
print("→ Vạch đỏ cắt ngang dòng vật chưa? Lệch thì sửa line_pos rồi chạy lại cell này.")

In [ ]:
# 🧠 Nạp LocateAnything-3B MỘT LẦN (float16 ~7GB). Dọn GPU trước để tránh CUDA OOM khi chạy lại cell.
import os, gc, torch
if "detector" in globals():          # chạy lại cell → giải phóng model cũ trên GPU
    del detector
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU trống {free/1e9:.1f}/{total/1e9:.1f} GB trước khi nạp")
    if free / 1e9 < 8:
        print("⚠️  GPU còn <8GB (model cũ KẸT — dọn cache KHÔNG đủ) → BẮT BUỘC:")
        print("    Run ▸ Restart & clear cell outputs, rồi chạy LẠI: setup → cell này (mỗi cái 1 LẦN).")
        raise SystemExit("Hãy RESTART kernel rồi chạy lại (đừng nạp model 2 lần).")

# Nếu SAU KHI restart vẫn OOM lúc nạp → bỏ dấu # dòng dưới (nạp thẳng GPU, ít VRAM nhất):
# os.environ["LA_DEVICE_MAP"] = "1"

from run_la_conveyor import build_fast_detector
detector = build_fast_detector("nvidia/LocateAnything-3B",
                               max_new_tokens=MAX_NEW_TOKENS, iou=0.5, max_boxes=80)
print("✅ Model sẵn sàng.")

In [ ]:
# ▶️ ĐẾM theo VẠCH (prompt đơn giản) → xuất video annotate + xem inline
import time, subprocess
from IPython.display import Video, display
from run_la_conveyor import run_video

OUT = os.path.join(WORK, "out_annot"); os.makedirs(OUT, exist_ok=True)
print(f"{'video':22}{'prompt':14}{'IN':>4}{'OUT':>5}{'tổng':>6}{'det/fr':>8}{'giây':>7}")
print("-" * 67)
for v in VIDEOS:
    q, stem = v["query"], os.path.splitext(os.path.basename(v["path"]))[0]
    mp4 = os.path.join(OUT, f"{stem}.mp4")
    t0 = time.time()
    r = run_video(detector, v["path"], q, orient=v["orient"], line_pos=v["line_pos"],
                  proc_width=PROC_WIDTH, max_frames=MAX_FRAMES, stride=STRIDE, save_video=mp4)
    print(f"{v['name'][:21]:22}{q[:13]:14}{r.in_count:>4}{r.out_count:>5}"
          f"{r.total_crossings:>6}{r.avg_detections:>8.1f}{time.time()-t0:>7.0f}")
    h264 = mp4.replace(".mp4", "_h264.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", mp4,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", h264], check=False)
    display(Video(h264 if os.path.exists(h264) else mp4, embed=True, width=560))
print("📁 Video annotate lưu tại:", OUT)
print("det/fr = 0 → đổi prompt ở cell dưới (so sánh) để chọn từ bắt nhiều nhất.")

In [ ]:
# 🔤 SO SÁNH PROMPT — chọn danh từ bắt được NHIỀU nhất (nhanh, ít frame)
import time
from run_la_conveyor import run_video

print(f"{'video':22}{'prompt':16}{'det/fr':>8}{'giây':>7}")
print("-" * 54)
for v in VIDEOS:
    for q in [v["query"]] + v.get("alts", []):
        t0 = time.time()
        r = run_video(detector, v["path"], q, orient=v["orient"], line_pos=v["line_pos"],
                      proc_width=PROC_WIDTH, max_frames=10, stride=2)   # ít frame = nhanh
        print(f"{v['name'][:21]:22}{q[:15]:16}{r.avg_detections:>8.1f}{time.time()-t0:>7.0f}")
print("→ Prompt có det/fr CAO nhất là tốt nhất; đưa nó vào 'query' ở cell cấu hình rồi chạy lại cell ĐẾM.")

### Đọc kết quả
- **det/fr > 0** = LocateAnything nhận ra vật. Nếu **= 0**: prompt chưa hợp → thử prompt
  khác ở cell **So sánh prompt** (carton box / package / box / tomato / fruit).
- **tổng** = số vật cắt vạch (tăng `MAX_FRAMES` nếu muốn nhiều hơn; vạch lệch → sửa `line_pos`).
- **Nguyên tắc prompt cho LA:** DÙNG danh từ đơn giản tiếng Anh; TRÁNH mô tả dài & tiếng Việt.
- LocateAnything **chậm** (~vài giây/frame) — bình thường. Nếu LA báo lỗi phiên bản: chạy
  `!pip install -q transformers==4.57.1` rồi chạy lại (đừng bấm Stop khi đang cài).